In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 1
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df

customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# 1) Count orders per customer (exclude special requests)
cust_order_counts = (
    orders
    .loc[
        ~orders["O_COMMENT"].str.contains(r"special[\S|\s]*requests"),
        "O_CUSTKEY"
    ]
    .value_counts()
)

# 2) Build distribution of counts for customers with ≥1 order
dist = (
    cust_order_counts
    .value_counts()
    .rename("CUSTDIST")
    .reset_index()
)
dist.columns = ["C_COUNT", "CUSTDIST"]

# compute how many customers had zero orders
zeros = len(customer) - len(cust_order_counts)
if zeros:
    # use concat instead of append (unsupported in cudf.DataFrame)
    zero_df = pd.DataFrame({"C_COUNT": [0], "CUSTDIST": [zeros]})
    dist = pd.concat([dist, zero_df], ignore_index=True)

# 3) Sort and assign to total
total = dist.sort_values(["CUSTDIST", "C_COUNT"], ascending=[False, False])